In [12]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Literal, Any, Annotated, List
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
import time
load_dotenv()

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')

In [13]:
# Define state
class CrashState(TypedDict):
    input: str 
    step1: str 
    step2: str 
    step3: str 
    

In [14]:
def step1(state: CrashState) -> CrashState:
    print('step 1 executed') 
    return {'step1': 'done', 'input': state['input']}

def step2(state: CrashState) -> CrashState:
    print('step 2 hanging... Now manually interrupt from the notebook toolbar (Stop Button)') 
    time.sleep(30) # Simulate long-running hang
    return {'step2': 'done'}

def step3(state: CrashState) -> CrashState:
    print('Step 3 executed')
    return {'step3': 'done'}

In [15]:
graph = StateGraph(CrashState)
graph.add_node('step_1', step1)
graph.add_node('step_2', step2)
graph.add_node('step_3', step3)

graph.add_edge(START, 'step_1')
graph.add_edge('step_1', 'step_2')
graph.add_edge('step_2', 'step_3')
graph.add_edge('step_3', END)

In [ ]:
checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)

workflow.invoke({'input': 'start'}, config={'configurable': {'thread_id': 'thread-1'}})

step 1 executed
step 2 hanging... Now manually interrupt from the notebook toolbar (Stop Button)
Step 3 executed


{'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}

: 